In [42]:
from scipy.optimize import fmin_tnc
import scipy.optimize as opt
import numpy as np
import os

In [43]:
def logsumexp(x: np.ndarray) -> float:
    """Compute log(sum(exp(x))) such that max(x) is used to avoid overflow"""
    M = np.max(x)
    return M + np.log(np.sum(np.exp(x - M)))

In [44]:
def letter_to_index(letter: str) -> int:
    """Convert a letter to an index (0 for 'a', 1 for 'b', ..., 25 for 'z')."""
    return ord(letter) - ord('a')

In [45]:
def load_data(path):
    sequences = []
    
    # Read the data file and parse each line
    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            parts = line.split()
            label_char = parts[1]
            feats = [float(v) for v in parts[5:]]
            
            # X stores all values of the features, and y stores the label index
            """It skips first 4 values (word_id, label, word, word_length) and takes the rest as features"""
            X = np.array([feats], dtype=np.float64)         # (1, d)
            y = np.array([letter_to_index(label_char)], dtype=np.int64)  # (1,)
            sequences.append((X, y))

    return sequences

In [46]:
def score_function(X, y, W, T):
    """Compute the score of a sequence (X,y) given model parameters W and T"""
    score = 0.0
    n = X.shape[0]
    for s in range(n-1):
        score += W[y[s]].dot(X[s]) + T[y[s], y[s+1]]

    score += W[y[n-1]].dot(X[n-1])

    return score

In [47]:
def compute_Z_forward(X, W, T):
    """Compute the partition function Z(X) for a sequence X given model parameters W and T"""
    
    # Using dynamic progrmming to compute the partition function Z(X)
    n, d = X.shape
    K, dW = W.shape

    alpha = np.zeros((n, K), dtype = np.float64)

    # Pre-compute the A matrix which is the score of each letter at each position without transition scores
    A = X @ W.T
    alpha[0, :] = A[0, :]

    """Iterate through the sequence and compute the alpha values using the A matrix and transition scores T.
    This is the forward pass for computing the partition function Z(X)"""
    for s in range(1, n):
        for j in range(K):
            prev_terms = alpha[s-1, :] + T[:, j]
            alpha[s, j] = logsumexp(prev_terms) + A[s, j]
        
    logZ = logsumexp(alpha[n-1, :])
    
    return logZ, alpha

In [48]:
def viterbi_decode_WTX(X, W, T):
    """
    Backward DP in log-space for CRF.
    Returns beta of shape (n, K).
    beta[s,i] = log sum of all suffix path scores from position s+1..n-1
                given y_s = i
    """

    n, d = X.shape
    K = W.shape[0]

    # Emission scores: A[s, j] = W[j] · X[s]
    A = X @ W.T  # (n, K)

    beta = np.zeros((n, K), dtype=np.float64)  # beta[n-1,:] = 0

    for s in range(n - 2, -1, -1):
        # for each current label i, sum over next label j
        # beta[s,i] = logsumexp_j( T[i,j] + A[s+1,j] + beta[s+1,j] )
        beta[s, :] = logsumexp(T + (A[s + 1, :] + beta[s + 1, :])[None, :], axis=1)
    
    delta = np.zeros((n, K), dtype=np.float64)
    DP = np.zeros((n, K), dtype=np.int32)

    delta[0, :] = A[0, :]
    for s in range(1, n):
        scores = delta[s - 1][:, None] + T  # (K, K): prev i -> curr j
        DP[s, :] = np.argmax(scores, axis=0)
        delta[s, :] = np.max(scores, axis=0) + A[s, :]

    y_pred = np.zeros(n, dtype=np.int32)
    y_pred[n - 1] = np.argmax(delta[n - 1, :])
    for s in range(n - 2, -1, -1):
        y_pred[s] = DP[s + 1, y_pred[s + 1]]

    return beta, y_pred

In [49]:
def compute_node_marginals(alpha, beta, logZ):
    """Compute the node marginals p(y_s | X) for each position s given alpha, beta, and logZ"""
    n, K = alpha.shape
    node_marginals = np.zeros((n, K), dtype=np.float64)

    # node_marginals measure the probability of each label at each position given the sequence X, and are computed using the alpha and beta values from the forward and backward passes, normalized by the partition function Z(X).
    for s in range(n):
        log_prob = alpha[s, :] + beta[s, :] - logZ
        
        """ probs are the unnormalized probabilities of each label at position s, computed by exponentiating the log probabilities. 
        To avoid numerical instability, we subtract the maximum log probability m from log_prob before exponentiating, 
        which ensures that we are working with smaller numbers and prevents overflow."""
        m = np.max(log_prob)
        probs = np.exp(log_prob - m)
        node_marginals[s, :] = probs / np.sum(probs)

    return node_marginals

In [50]:
def compute_edge_marginals(X, W, T, alpha, beta, logZ):
    """Compute the edge marginals p(y_s, y_{s+1} | X) for each position s given X, W, T, alpha, beta, and logZ"""
    n, d = X.shape
    K = W.shape[0]
    edge_marginals = np.zeros((n-1, K, K), dtype=np.float64)

    # edge_marginals measure the joint probability of pairs of labels at adjacent positions given the sequence X, and are computed using the alpha and beta values from the forward and backward passes, along with the model parameters W and T, normalized by the partition function Z(X).
    A = X @ W.T

    # log_probs are the unnormalized log probabilities of each pair of labels at positions s and s+1, computed using the alpha values at position s, the transition scores T, the A matrix for the scores of each label at each position, and the beta values at position s+1.
    for s in range(n-1):
        log_probs = (alpha[s][:, None]
                     + T
                     + A[s+1][None, :]
                     + beta[s+1][None, :]
                     - logZ)

        # To avoid numerical overflow, subtract the maximum log probability from each log probability before exponentiating
        m = np.max(log_probs)
        probs = np.exp(log_probs - m)
        edge_marginals[s] = probs / np.sum(probs)

    return edge_marginals

In [51]:
def compute_logp_y_given_X(X, y, W, T):

    # Compute the log probability of a sequence (X,y) given model parameters W and T using the score function and the partition function Z(X)
    logZ, _ = compute_Z_forward(X, W, T)
    score = score_function(X, y, W, T)
    return score - logZ

In [52]:
def gradient_loglikelihood(X, y, W, T):
    """Compute the gradient of the log likelihood with respect to W and T for a single sequence (X,y)"""
    n, d = X.shape
    K = W.shape[0]

    # Compute the node and edge marginals
    logZ, alpha = compute_Z_forward(X, W, T)
    beta, y_pred = viterbi_decode_WTX(X, W, T)
    node_marginals = compute_node_marginals(alpha, beta, logZ)
    edge_marginals = compute_edge_marginals(X, W, T, alpha, beta, logZ)

    # Initialize gradients
    grad_W = np.zeros_like(W)
    grad_T = np.zeros_like(T)

    # Compute the gradient with respect to W
    for s in range(n):
        grad_W[y[s]] += X[s]  
        grad_W -= node_marginals[s][:, None] * X[s]

    # Compute the gradient with respect to T
    for s in range(n-1):
        grad_T[y[s], y[s+1]] += 1
        grad_T -= edge_marginals[s]

    return grad_W, grad_T, y_pred

In [53]:
def get_true_labels(word_list):
    """
    Extract true label sequences from word_list.
    Returns a list of Python lists.
    """
    true_labels = []
    for X, y in word_list:
        y = np.asarray(y).reshape(-1).astype(int).tolist()
        true_labels.append(y)
    return true_labels

In [54]:
def compare(predictions, true_labels):
    """Compare predicted labels with true labels and compute accuracy"""
    correct = 0
    total = 0

    for pred, true in zip(predictions, true_labels):
        pred = list(pred)
        true = list(true)

        L = min(len(pred), len(true))  # avoid index mismatch
        for i in range(L):
            if pred[i] == true[i]:
                correct += 1
            total += 1

    if total == 0:
        return 0.0

    return correct / total

In [55]:
def get_crf_obj(word_list, W, T, c):
    """ Compute the negative log-likelihood of the CRF model with L2 regularization 
    given a list of sequences (word_list), model parameters W and T, and regularization strength c"""
    total_loglik = 0.0
    grad_W = np.zeros_like(W)
    grad_T = np.zeros_like(T)

    for X, y in word_list:
        score = score_function(X, y, W, T)
        logZ, _ = compute_Z_forward(X, W, T)
        dW, dT, _ = gradient_loglikelihood(X, y, W, T)  # gradient of (score - logZ)
        grad_W += dW
        grad_T += dT
        total_loglik += score - logZ

    # Negative log-likelihood + L2 regularization
    # -c * total_loglik + 0.5(||W||^2 + ||T||^2)
    obj = -c * total_loglik + 0.5 * (np.sum(W*W) + np.sum(T*T))

    return obj, grad_W, grad_T

In [62]:
def crf_obj(x, word_list, c):
    k, d = 26, 128
    x = np.asarray(x).ravel()
    W = x[:k*d].reshape(k, d)
    T = x[k*d:].reshape(k, k)
    
    # updates the value of grad_W and grad_T based on the current W and T, and computes the objective function value
    f, grad_W, grad_T = get_crf_obj(word_list, W, T, c)
    grad_W = -c * grad_W + W
    grad_T = -c * grad_T + T

    print("Optimal objective value:", f)

    g = np.concatenate([grad_W.reshape(-1), grad_T.reshape(-1)])
    return f, g

In [57]:
def change_directory(txt_file):
    current_dir = os.getcwd()
    project_root = os.path.dirname(current_dir)

    result_path = os.path.join(project_root, "result", txt_file)

    os.makedirs(os.path.dirname(result_path), exist_ok=True)

    return result_path

In [58]:
def crf_test(x, word_list):
    """
    Evaluate a trained CRF model on word_list and save predictions vs labels.
    """
    import numpy as np

    x = np.asarray(x).ravel()
    k, d = 26, 128

    # Unpack parameters safely
    assert x.size == k*d + k*k, f"x has size {x.size}, expected {k*d + k*k}"
    W = x[:k*d].reshape(k, d)
    T = x[k*d:].reshape(k, k)

    y_hat = []

    # compute predictions for each sequence in word_list using the Viterbi algorithm, and store them in y_hat
    for idx, (X, y) in enumerate(word_list):
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1)

        # get prediction (whatever the function returns)
        _, _, y_pred = gradient_loglikelihood(X, y, W, T)

        # normalize prediction to 1D python list of ints
        y_pred = np.asarray(y_pred).reshape(-1).astype(int).tolist()
        y_hat.append(y_pred)

    true_label_of_word_list = [
        np.asarray(y).reshape(-1).astype(int).tolist() for (X, y) in word_list
    ]

    accuracy = compare(y_hat, true_label_of_word_list)
    print(f"Accuracy = {accuracy}\n")
    print(f"Num sequences: {len(word_list)} ")

    path = change_directory("prediction.txt")

    # store the predicted labels in a file predictions.txt
    with open(path, "w") as f:
      for ypred in y_hat:
        for label in ypred:
            f.write(f"{label + 1}\n")

    return accuracy

In [59]:
def ref_optimize(train_data, test_data, c):
    print('Training CRF ... c = {} \n'.format(c))

    # Initial value of the parameters W and T, stored in a vector
    x0 = np.zeros(128*26 + 26**2, dtype=np.float64)

    # Start the optimization
    result = opt.fmin_tnc(crf_obj, x0, args = [train_data, c], maxfun=100,
                          ftol=1e-3, disp=5)
    
    model  = result[0] 
    
    # save the optimized parameters of W and T to a file solution.txt
    path = change_directory("solution.txt")

    with open(path, "w") as f:
        for value in model:
            f.write(f"{value}\n")

    accuracy = crf_test(model, test_data)

    print("model norm:", np.linalg.norm(model))
    print("max abs param:", np.max(np.abs(model)))
    print('CRF test accuracy for c = {}: {}'.format(c, accuracy))
    return accuracy

In [60]:
def main():
    # Load the data
    current_dir = os.getcwd()

    # Go up one level to project_root
    project_root = os.path.dirname(current_dir)

    data_path = os.path.join(project_root, "data", "train.txt")
    train_data = load_data(data_path)

    data_test_path = os.path.join(project_root, "data", "test.txt")
    test_data = load_data(data_test_path)

    # Regularization strength
    c = 1000

    # Train the CRF and evaluate on the test set
    accuracy = ref_optimize(train_data, test_data, c)
    error = 1 - accuracy
    print('CRF test error for c = {}: {}'.format(c, error))

In [63]:
if __name__ == "__main__":
    main()


Training CRF ... c = 1000 

Optimal objective value: 84557379.45122722
Optimal objective value: 84557379.22653522
Optimal objective value: 84557379.33843991
Optimal objective value: 84557379.43145391
Optimal objective value: 84557379.44716232
Optimal objective value: 40912984.95342236
Optimal objective value: 49333276.86332337
Optimal objective value: 42322133.1777822
Optimal objective value: 41092028.168234974
Optimal objective value: 40890601.584637545
Optimal objective value: 40890601.2391253
Optimal objective value: 40890601.39376116
Optimal objective value: 40890601.436335646
Optimal objective value: 40890601.41936711
Optimal objective value: 40890601.47246155
Optimal objective value: 40890601.46663739
Optimal objective value: 31438461.897522334
Optimal objective value: 32256898.866172884
Optimal objective value: 30814659.55014744
Optimal objective value: 30814659.277452484
Optimal objective value: 30814659.49804252
Optimal objective value: 28874289.209480878
Optimal objective val